In [ ]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mysql.connector

In [ ]:
conn = mysql.connector.connect(
    host='localhost',
    user='root',
    password='xyz',
    database='olist_data'
)
cursor = conn.cursor()

**question 1** **. Which customers placed the most orders overall**.


In [ ]:
query = '''SELECT 
    c.customer_id,
    COUNT(o.order_id) AS order_count
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id
ORDER BY order_count DESC
LIMIT 10; '''
cursor.execute(query)
result = cursor.fetchall()
result
data = pd.DataFrame(result,columns = ["customer_id","count"])
data

# **question 2** **Find repeat customers placed more than 1 order**.


In [ ]:
query = '''SELECT 
    c.customer_id,
    COUNT(o.order_id) AS order_count
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id
having order_count>1
ORDER BY order_count DESC
LIMIT 10; '''
cursor.execute(query)
result = cursor.fetchall()
result
data = pd.DataFrame(result,columns = ["customer_id","count"])
data.head(2)

# **question 3** **What is the average time between a customer’s first and second order?**.

In [ ]:
query = '''SELECT AVG(DATEDIFF(order_delivered_customer_date, order_delivered_carrier_date)) AS avg_time
FROM orders;'''
cursor.execute(query)
result = cursor.fetchall()
result
data = pd.DataFrame(result,columns = ["avg_time"])
data

# **question 4** ** Identify customers who placed an order but never received it (cancelled or not delivered)**

In [ ]:
query = '''select customer_id , order_status from orders
WHERE (order_status = 'canceled' OR order_status = 'unavailable')'''
cursor.execute(query)
result = cursor.fetchall()
data = pd.DataFrame(result,columns = ["customer_id","order_status"])
data

# **question 5 ** **Compute customer lifetime value (CLV) (total money spent per customer)**.


In [ ]:
query = ''' select o.customer_id , round(sum(ot.price + ot.freight_value)) as total_amount from orders o
join order_items ot on o.order_id = ot.order_id
group by o.customer_id 
order by total_amount desc '''
cursor.execute(query)
result = cursor.fetchall()
data = pd.DataFrame(result,columns = ["customer_id","total_amount"])
data.head(5)


# **question 6 ** ** Which product categories have the highest revenue vs highest number of orders? (compare popularity vs profitability)**.

In [ ]:
query = '''
SELECT 
    ot.product_id, 
    COUNT(ot.product_id) AS total_buy,
    ROUND(SUM(ot.price + ot.freight_value)) AS total_amount
FROM order_items ot
JOIN orders o ON ot.order_id = o.order_id
GROUP BY ot.product_id
ORDER BY total_amount DESC
LIMIT 10;
'''

cursor.execute(query)
result = cursor.fetchall()
data = pd.DataFrame(result, columns=["product_id", "total_buy", "total_amount"])
data.head(5)


x = np.arange(len(data['product_id']))
width = 0.4

fig, ax1 = plt.subplots(figsize=(12,6))

# Total Orders (left y-axis)
ax1.bar(x - width/2, data['total_buy'], width, color='skyblue', label='Total Orders')
ax1.set_ylabel('Total Orders', color='skyblue')
ax1.tick_params(axis='y', labelcolor='skyblue')

# Total Revenue (right y-axis)
ax2 = ax1.twinx()
ax2.bar(x + width/2, data['total_amount'], width, color='orange', label='Total Revenue')
ax2.set_ylabel('Total Revenue', color='orange')
ax2.tick_params(axis='y', labelcolor='orange')

plt.xticks(x, data['product_id'], rotation=45)
plt.title('Top 10 Products: Popularity vs Revenue')
fig.tight_layout()
plt.show()

# **question 7** ** Find the most refunded or canceled product categories**.

In [ ]:
query = '''SELECT 
    p.product_category_name,
    COUNT(p.product_category_name) AS count_times,
    o.order_status
FROM orders o
JOIN order_items ot ON o.order_id = ot.order_id
JOIN products p ON ot.product_id = p.product_id
WHERE o.order_status = 'canceled' OR o.order_status = 'unavailable'
GROUP BY p.product_category_name, o.order_status
order by count_times desc'''
cursor.execute(query)
result = cursor.fetchall()
data = pd.DataFrame(result, columns=["category_name", "count_times", "order_status"])
data.head(5)


**question** **Find the average delivery time for each product category**.

In [ ]:
query = '''SELECT 
    p.product_id,
    SEC_TO_TIME(AVG(TIME_TO_SEC(TIME(o.order_delivered_customer_date)))) AS avg_delivery_time
FROM order_items ot
JOIN orders o   ON ot.order_id = o.order_id
JOIN products p ON ot.product_id = p.product_id
GROUP BY p.product_id
order by avg_delivery_time desc'''
cursor.execute(query)
result = cursor.fetchall()
data = pd.DataFrame(result, columns=["product_id", "avg_delivery_time"])
data.head(5)

# **question** ** Which categories have high freight cost as a % of order price**.

In [ ]:
query = '''SELECT 
    p.product_category_name,
    SUM(ot.freight_value) / SUM(ot.price) * 100 AS freight_pct
FROM order_items ot
JOIN products p ON ot.product_id = p.product_id
GROUP BY p.product_category_name
ORDER BY freight_pct DESC;
''' 
cursor.execute(query)
result = cursor.fetchall()
data = pd.DataFrame(result, columns=["category_name", "freight_pct"])
data.head(5)

#  Question 11. Which sellers generate the highest revenue?.

In [ ]:
query = '''select s.seller_id, round(sum(ot.price)) as total_price from order_items ot 
join sellers s on ot.seller_id = s.seller_id 
group by seller_id 
order by total_price desc'''
cursor.execute(query)
result = cursor.fetchall()
data = pd.DataFrame(result, columns=["seller_id", "total_price"])
data= data.head(5)
plt.figure(figsize=(12,6))
plt.bar(data['seller_id'], data['total_price'], color = ['darkred', 'red', 'orange', 'green', 'blue'])
plt.xticks(rotation=90) 
plt.title('Top Sellers')
plt.xlabel('Seller Id')
plt.ylabel('Total_price')
plt.show()


# Find sellers with longest average delivery delays.

In [ ]:
query = '''SELECT 
    ot.seller_id,
    AVG(TIMESTAMPDIFF(DAY, o.order_purchase_timestamp, o.order_delivered_customer_date)) AS avg_delivery_delay_days
FROM order_items ot
JOIN orders o ON ot.order_id = o.order_id
WHERE o.order_delivered_customer_date IS NOT NULL
GROUP BY ot.seller_id
ORDER BY avg_delivery_delay_days DESC
LIMIT 10;  
'''
cursor.execute(query)
result = cursor.fetchall()
data = pd.DataFrame(result, columns=["seller_id", "avg_delivey_time"])
data= data.head(5)
data

# Rank sellers by customer satisfaction (using review_score).

In [ ]:
query = ''' select ot.seller_id,sum(orr.review_score) as review_score,rank() over(order by sum(orr.review_score) desc) as rankk from order_items ot 
join order_reviews orr on ot.order_id = orr.order_id
group by ot.seller_id
order by rankk  
'''
cursor.execute(query)
result = cursor.fetchall()
data = pd.DataFrame(result, columns=["seller_id", "review_score","rankk"])
data= data.head(5)
data

# Which sellers have most cancellations or refunds?

In [ ]:
query = '''
SELECT 
    ot.seller_id, 
    o.order_status,
    COUNT(o.order_status) AS total_count
FROM orders o 
JOIN order_items ot ON o.order_id = ot.order_id 
WHERE o.order_status = "canceled"
GROUP BY ot.seller_id 
ORDER BY total_count DESC
'''

cursor.execute(query)
result = cursor.fetchall()


data = pd.DataFrame(result, columns=["seller_id", "order_status", "total_count"])


data = data.head(5)

plt.figure(figsize=(12,6))
bars = plt.bar(data["seller_id"], data["total_count"], color=['darkred', 'red', 'orange', 'green'])  


for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.5, f'{int(height)}', ha='center', va='bottom')

plt.xticks(rotation=90)
plt.title('Cancellations per Seller')
plt.xlabel('Seller ID')
plt.ylabel('Number of Cancellations')
plt.show()


# Find sellers who sell in multiple categories vs sellers in single category.

In [ ]:
query = '''
SELECT 
    ot.seller_id, 
    COUNT(DISTINCT p.product_category_name) AS product_count,
    CASE 
        WHEN COUNT(DISTINCT p.product_category_name) = 1 
            THEN 'Single Category Seller'
        ELSE 'Multiple Category Seller'
    END AS seller_type
FROM order_items ot
JOIN products p 
    ON p.product_id = ot.product_id
GROUP BY ot.seller_id
ORDER BY product_count desc'''
cursor.execute(query)
result = cursor.fetchall()
data = pd.DataFrame(result, columns=["seller_id", "product_count", "seller_status"])

data = data["seller_status"].value_counts()
plt.figure(figsize=(12,6))
bars = plt.bar(data.index, data.values, color=['red','green'])  


for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, height + 0.5, f'{int(height)}', ha='center', va='bottom')

plt.xticks(rotation=90)
plt.title('Cancellations per Seller')
plt.xlabel('Seller ID')
plt.ylabel('Number of Cancellations')
plt.show()


#  Compute the average delivery delay = delivered_date − estimated_delivery_date

In [ ]:
query = '''
SELECT 
    AVG(ABS(DATEDIFF(order_delivered_customer_date, order_estimated_delivery_date))) AS avg_delivery
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL;
'''

cursor.execute(query)
result = cursor.fetchall()
data = pd.DataFrame(result, columns=["avg_delivery_delay_days"])
data = data.head(5)
data


#  Find the % of orders delivered late each month


In [ ]:
query = '''SELECT 
    MONTH(order_delivered_customer_date) AS delivery_month,
    COUNT(*) AS total_orders,
    SUM(CASE 
            WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 1
            ELSE 0
        END) AS late_orders,
    ROUND(
        SUM(CASE 
                WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 1
                ELSE 0
            END) * 100.0 / COUNT(*), 2
    ) AS late_percentage
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL
GROUP BY MONTH(order_delivered_customer_date)
ORDER BY delivery_month;'''

cursor.execute(query)
result = cursor.fetchall()
data = pd.DataFrame(result, columns=["delivery_month","total_orders","late_orders","late_percentage"])
data = data.head(5)
data

#  Which states/cities have the longest average delivery times?

In [ ]:
query = ''' 
 select avg(datediff(o.order_delivered_customer_date,o.order_estimated_delivery_date)) 
 as delivery_time, c.customer_state from customers c
 join orders o on c.customer_id = o.customer_id
 where datediff(o.order_delivered_customer_date,o.order_estimated_delivery_date) > 0
 group by c.customer_state
 order by delivery_time desc '''

cursor.execute(query)
result = cursor.fetchall()
data = pd.DataFrame(result, columns=["delivery_time","customer_state"])
data = data.head(5)
data

# Compare freight cost vs delivery distance.

In [ ]:
query = '''SELECT 
    order_id,
    DATEDIFF(order_delivered_customer_date, order_estimated_delivery_date) AS delivery_delay,
    CASE 
        WHEN DATEDIFF(order_delivered_customer_date, order_estimated_delivery_date) > 0 THEN 'Not Good'
        WHEN DATEDIFF(order_delivered_customer_date, order_estimated_delivery_date) = 0 THEN 'Good'
        WHEN DATEDIFF(order_delivered_customer_date, order_estimated_delivery_date) < 0 THEN 'Excellent'
    END AS delivery_performance
FROM orders
WHERE order_delivered_customer_date IS NOT NULL
  AND order_estimated_delivery_date IS NOT NULL;'''

cursor.execute(query)
result = cursor.fetchall()


data = pd.DataFrame(result, columns=["order_id","delivery_delay","delivery_performance"])

data_counts = data['delivery_performance'].value_counts()


plt.figure(figsize=(8,5))
plt.bar(data_counts.index, data_counts.values, color=['red', 'orange', 'green'])
plt.title('Orders by Delivery Performance')
plt.xlabel('Delivery Performance')
plt.ylabel('Number of Orders')
plt.show()


# Which payment type contributes the most to revenue?

In [ ]:
query = '''SELECT 
    payment_type,
    round(SUM(payment_value)) AS total_payment
FROM order_payment
WHERE payment_type != 'not_defined'
GROUP BY payment_type
ORDER BY total_payment DESC;'''
cursor.execute(query)
result = cursor.fetchall()


data = pd.DataFrame(result, columns=["payment_type","total_payment"])
plt.figure(figsize=(8,8))
plt.pie(
    data['total_payment'],
    labels=None,          
    autopct='%1.1f%%',    
    startangle=90,
    colors=['skyblue', 'orange', 'green', 'red', 'purple'] 
)


plt.legend(
    data['payment_type'], 
    title="Payment Types", 
    bbox_to_anchor=(1, 0.5),
    loc='center left'
)

plt.title('Payment Distribution by Payment Type')
plt.tight_layout()
plt.show()


#  Find the average number of installments per payment method

In [ ]:
query = ''' select payment_type,round(avg(payment_installments)) from order_payment
group by payment_type '''
cursor.execute(query)
result = cursor.fetchall()


data = pd.DataFrame(result, columns=["payment_type","avg_payment_installemnts"])




plt.figure(figsize=(8,5))
plt.bar(data["payment_type"], data["avg_payment_installemnts"], color=['red', 'orange', 'green'])
plt.title('Orders by Delivery Performance')
plt.xlabel('Delivery Performance')
plt.ylabel('Number of Orders')
plt.show()


# Which orders had partial payments (multiple payment records for same order)?

In [ ]:
query = '''SELECT 
    order_id, 
    COUNT(payment_type) AS payment_count
FROM order_payment
GROUP BY order_id
HAVING COUNT(payment_type) > 1;'''

cursor.execute(query)
result = cursor.fetchall()


data = pd.DataFrame(result, columns=["order_id","payment_type"])
data

# Calculate the % of total revenue paid using Credit Card, Boleto, Debit Card, Voucher.

In [ ]:
query = '''
SELECT 
    payment_type,
    ROUND(SUM(payment_value)) AS total_payment
FROM order_payment
WHERE payment_type != 'not_defined'
GROUP BY payment_type
ORDER BY total_payment DESC;
'''
cursor.execute(query)
result = cursor.fetchall()

data = pd.DataFrame(result, columns=["payment_type","total_payment"])


data["percentage"] = (data["total_payment"] / data["total_payment"].sum()) * 100

print(data)

# Plot pie chart
plt.figure(figsize=(8,8))
plt.pie(
    data['total_payment'],
    labels=None,          
    autopct='%1.1f%%',    
    startangle=90,
    colors=['skyblue', 'orange', 'green', 'red', 'purple'] 
)

plt.legend(
    data['payment_type'], 
    title="Payment Types", 
    bbox_to_anchor=(1, 0.5),
    loc='center left'
)

plt.title('Payment Distribution by Payment Type')
plt.tight_layout()
plt.show()


#  Find customers who used different payment methods across orders.

In [ ]:
query = '''SELECT 
    o.customer_id,
    COUNT(DISTINCT op.payment_type) AS payment_methods_used
FROM orders o
JOIN order_payment op 
    ON op.order_id = o.order_id
GROUP BY o.customer_id
HAVING COUNT(DISTINCT op.payment_type) > 1
ORDER BY payment_methods_used DESC;'''
cursor.execute(query)
result = cursor.fetchall()

data = pd.DataFrame(result, columns=["customer_id","payment_method_count"])



data

#  For each month, calculate the running total revenue (cumulative sum)

In [ ]:
query = ''' SELECT 
    MONTH(o.order_purchase_timestamp) AS months,
    ROUND(SUM(ot.price)) AS total_revenue,
    SUM(ROUND(SUM(ot.price))) OVER (
        ORDER BY MONTH(o.order_purchase_timestamp)
    ) AS running_revenue
FROM orders o
JOIN order_items ot 
    ON o.order_id = ot.order_id
GROUP BY MONTH(o.order_purchase_timestamp)
ORDER BY months;
'''                
cursor.execute(query)
result = cursor.fetchall()

data = pd.DataFrame(result, columns=["months","total_revenue","running_revenue"])
data
plt.bar(data['months'], data['total_revenue'], color = [
    'skyblue',   
    'orange',    
    'green',     
    'red',       
    'purple',    
    'gold',      
    'pink',      
    'lightgreen',
    'brown',     
    'cyan',      
    'magenta',   
    'navy'       
]
, label='Monthly Revenue')


plt.plot(data['months'], data['running_revenue'], color='red', marker='o', label='Running Total Revenue')


plt.xlabel('Month')
plt.ylabel('Revenue')
plt.title('Monthly Revenue and Running Total')
plt.xticks(rotation=45)  # Rotate month labels
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# For each customer, find the rank of their orders by purchase date.

In [ ]:
query = '''SELECT 
    customer_id,
    order_id,
    order_purchase_timestamp,
    ROW_NUMBER() OVER (
        PARTITION BY customer_id
        ORDER BY order_purchase_timestamp
    ) AS order_rank
FROM orders
ORDER BY customer_id, order_rank;
''' 
cursor.execute(query)
result = cursor.fetchall()

data = pd.DataFrame(result, columns=["customer_id","order_id","order_purchase_timestamp","order_rank"])
data


In [ ]:
query = '''WITH first_orders AS (
    
    SELECT customer_id,
           DATE_FORMAT(MIN(order_purchase_timestamp), '%Y-%m') AS cohort_month
    FROM orders
    GROUP BY customer_id
),
orders_with_cohort AS (
    -- Step 2: Attach cohort month to all orders
    SELECT o.customer_id,
           f.cohort_month,
           DATE_FORMAT(o.order_purchase_timestamp, '%Y-%m') AS order_month
    FROM orders o
    JOIN first_orders f
      ON o.customer_id = f.customer_id
)
-- Step 3: Count orders month-wise per cohort
SELECT cohort_month,
       order_month,
       COUNT(DISTINCT customer_id) AS active_customers
FROM orders_with_cohort
GROUP BY cohort_month, order_month
ORDER BY cohort_month, order_month;'''
cursor.execute(query)
result = cursor.fetchall()

data = pd.DataFrame(result, columns=["cohort_month","order_month","active_customers"])
data

# Find the top 3 products in each category by revenue

In [ ]:
query = '''SELECT *
FROM (
    SELECT 
        ot.product_id,
        p.product_category_name,
        ROUND(SUM(ot.price), 2) AS total_price,
        RANK() OVER (
            PARTITION BY p.product_category_name 
            ORDER BY ROUND(SUM(ot.price), 2) DESC
        ) AS ranking
    FROM order_items ot
    JOIN products p 
        ON ot.product_id = p.product_id
    GROUP BY ot.product_id, p.product_category_name
) ranked_products
WHERE ranking <= 3
ORDER BY product_category_name, ranking;

 
''' 
cursor.execute(query)
result = cursor.fetchall()

data = pd.DataFrame(result, columns=["product_id","product_category_name","total_price","ranking"])
data


# Identify seasonality trends: which months consistently have higher sales

In [ ]:
query = '''select month(o.order_purchase_timestamp) as order_month,round(sum(ot.price)) as price from order_items ot
join orders o on ot.order_id = o.order_id
group by  month(o.order_purchase_timestamp) 
order by order_month 
''' 
cursor.execute(query)
result = cursor.fetchall()

data = pd.DataFrame(result, columns=["order_month","price"])
plt.figure(figsize=(10,6))
plt.bar(data['order_month'], data['price'], color = [
    'red', 'blue', 'green', 'orange', 'purple', 'cyan', 
    'magenta', 'yellow', 'brown', 'pink', 'lime', 'navy'
])
plt.plot(data['order_month'], data['price'], color='black', marker='o', linewidth=2, label='Trend Line')


plt.xticks(range(1,13)) 
plt.xlabel('Month')
plt.ylabel('Total Sales')
plt.title('Monthly Sales Bar Chart (Seasonality)')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()